💡 **Environment:** `clamp-analyses`

# ARCHS4 consensus LVs: reproduction of the legacy sets

The legacy LV cache (`Z_all.npz`) and per-run noise screens were never found, so the two published consensus sets were rebuilt from scratch: the 10 CLAMPfull runs were exported from their `.rds` files, technical flags were re-thresholded from the legacy `noise_flags.csv`, and the **unchanged** `cluster_metalv.py` (complete linkage, cut at cos ≥ 0.5) was run on them.

- `cl_c050_m4_r015`: `min_models` 4, `noise_r2` 0.15, compared with `Z_4reps.csv`
- `cl_c050_m6_r025`: `min_models` 6, `noise_r2` 0.25, compared with `Z_6reps.csv`

A set passes when its metaLV names, gene order and loadings (max |ΔZ| < 1e-6) equal the legacy Z, and every metaLV has the legacy member LVs.

This is an internal check of the inputs; it will be removed once the consensus evaluation is done (PLAN step 4.3).


In [ ]:
suppressPackageStartupMessages({
  library(data.table)
  library(ggplot2)
})
sets <- unlist(snakemake@params[["sets"]])
min_models <- setNames(unlist(snakemake@params[["min_models"]]), sets)


## Reproduction checks

In [ ]:
reports <- rbindlist(lapply(snakemake@input[["reports"]], fread))
as.data.frame(reports[, .(set, n_ours, n_external, names_equal, genes_equal, max_abs_dZ,
                          byte_identical, members_equal, pass)])


## Clusters by number of runs spanned

All clusters from the complete-linkage cut; those spanning at least `min_models` runs (right of the dashed line) become metaLVs.

In [ ]:
counts <- rbindlist(lapply(seq_along(sets), function(i)
  fread(snakemake@input[["counts"]][[i]])[, set := sets[i]]))
counts[, kept := n_models >= min_models[set]]
counts[, set := factor(set, levels = sets)]
options(repr.plot.width = 10, repr.plot.height = 4)
ggplot(counts, aes(factor(n_models), count, fill = kept)) +
  geom_col() +
  geom_vline(data = data.table(set = factor(sets, levels = sets), x = min_models - 0.5),
             aes(xintercept = x), linetype = "dashed") +
  facet_wrap(~set) +
  scale_fill_manual(values = c(`FALSE` = "grey70", `TRUE` = "steelblue"),
                    labels = c(`FALSE` = "dropped", `TRUE` = "kept (metaLV)")) +
  labs(x = "Runs spanned (n_models)", y = "Clusters", fill = NULL) +
  theme_bw()


In [ ]:
as.data.frame(counts[, .(min_models = min_models[as.character(set[1])], clusters = sum(count),
                         metaLVs = sum(count[kept])), by = set])


In [ ]:
cat(if (all(reports$pass)) "All sets reproduce the legacy consensus sets exactly." else
    paste("FAILED:", paste(reports[pass == FALSE, set], collapse = ", ")), "\n")
